In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import xlsxwriter

In [2]:
metrics_dir = '/Users/u0186653/Desktop/research/dingen/deepnir/outputs/xgb'
components = ['ADF', 'Ash', 'Crude Fat', 'Crude Fib.', 'DM', 'NDF', 'Protein', 'Starch']

In [3]:
# Collect all data
all_component_data = []

for comp in components:
    csv_pattern = f"{comp.replace(' ', '_').replace('.', '')}_regr_XGB_.csv"
    all_data = []

    # Process per-crop results
    for freq_type in ['58freqs', 'all_freqs']:
        for crop in ['Barley', 'Corn', 'Rapeseed', 'Soybean', 'Wheat']:
            csv_file = os.path.join(metrics_dir, 'performance', freq_type, crop, csv_pattern)
            if os.path.exists(csv_file):
                df = pd.read_csv(csv_file)
                df['Crop'] = crop
                df['Freq'] = freq_type
                all_data.append(df)

        # Process multi-dataset results
        multi_path = os.path.join(metrics_dir, 'performance', freq_type, 'multi_dataset_training')
        if os.path.exists(multi_path):
            for subfolder in os.listdir(multi_path):
                sub_path = os.path.join(multi_path, subfolder)
                if os.path.isdir(sub_path):
                    csv_file = os.path.join(sub_path, csv_pattern)
                    if os.path.exists(csv_file):
                        df = pd.read_csv(csv_file)
                        df['Crop'] = f"Multi_{subfolder}"
                        df['Freq'] = freq_type
                        all_data.append(df)

    if not all_data:
        print(f"No data found for {comp}")
        continue

    # Combine data for this component
    data = pd.concat(all_data, ignore_index=True)
    suffix = comp.split()[-1].rstrip('.')
    data['Component'] = comp
    data['R2'] = data[f'val_R2_{suffix}']
    data['SEP'] = data[f'val_SEP_{suffix}']
    data['Std_SEP'] = data[f'val_std_SEP_{suffix}']
    data['Bias'] = data[f'val_Bias_{suffix}']

    # Keep only essential columns
    clean_data = data[['Component', 'Crop', 'Freq', 'R2', 'SEP', 'Std_SEP', 'Bias']]
    all_component_data.append(clean_data)

# Save to Excel with formatting
if all_component_data:
    with pd.ExcelWriter("./unified_metrics.xlsx", engine='xlsxwriter') as writer:
        # Combine all components
        unified_df = pd.concat(all_component_data, ignore_index=True)
        
        # Write to Excel
        unified_df.to_excel(writer, sheet_name='Summary', index=False)
        
        # Format for better readability
        workbook = writer.book
        worksheet = writer.sheets['Summary']
        
        # Add number formats
        float_fmt = workbook.add_format({'num_format': '0.000'})
        percent_fmt = workbook.add_format({'num_format': '0.0%'})
        
        # Apply formatting
        worksheet.set_column('A:A', 12)  # Component
        worksheet.set_column('B:B', 15)  # Crop
        worksheet.set_column('C:C', 8)   # Freq
        worksheet.set_column('D:D', 10, percent_fmt)  # R2
        worksheet.set_column('E:E', 10, float_fmt)    # SEP
        worksheet.set_column('F:F', 12, float_fmt)    # Std_SEP
        worksheet.set_column('G:G', 10, float_fmt)    # Bias

    print(f"✅ Unified table saved to ./unified_metrics.xlsx'")
else:
    print("❌ No data to save.")

No data found for Crude Fat
No data found for Crude Fib.
✅ Unified table saved to ./unified_metrics.xlsx'


In [ ]:
for comp in components:
    csv_pattern = f"{comp.replace(' ', '_').replace('.', '')}_regr_XGB_.csv"
    all_data = []

    for freq_type in ['58freqs', 'all_freqs']:
        for crop in ['Barley', 'Corn', 'Rapeseed', 'Soybean', 'Wheat']:
            csv_file = os.path.join(metrics_dir, 'performance', freq_type, crop, csv_pattern)
            if os.path.exists(csv_file):
                df = pd.read_csv(csv_file)
                df['Crop'] = crop
                df['Freq'] = freq_type
                all_data.append(df)

        multi_path = os.path.join(metrics_dir, 'performance', freq_type, 'multi_dataset_training')
        if os.path.exists(multi_path):
            for subfolder in os.listdir(multi_path):
                sub_path = os.path.join(multi_path, subfolder)
                if os.path.isdir(sub_path):
                    csv_file = os.path.join(sub_path, csv_pattern)
                    if os.path.exists(csv_file):
                        df = pd.read_csv(csv_file)
                        df['Crop'] = f"Multi_{subfolder}"
                        df['Freq'] = freq_type
                        all_data.append(df)

    if not all_data:
        print(f"No data found for {comp}")
        continue

    data = pd.concat(all_data, ignore_index=True)
    suffix = comp.split()[-1].rstrip('.')
    r2_col = f'val_R2_{suffix}'
    sep_col = f'val_SEP_{suffix}'
    std_sep_col = f'val_std_SEP_{suffix}'
    bias_col = f'val_Bias_{suffix}'

    # Normalize: higher R² is good, lower SEP/Bias are good
    data['R²'] = data[r2_col]
    data['SEP'] = 1 - (data[sep_col] / data[sep_col].max())  # Invert
    # data['std/SEP'] = 1 - (data[std_sep_col] / data[std_sep_col].max())  # Invert
    data['Bias'] = 1 - (data[bias_col].abs() / data[bias_col].abs().max())  # Invert

    # # After computing std/SEP
    # threshold_raw = 3.0
    # # Cap values below threshold (or set to NaN to exclude)
    # data['std/SEP_raw'] = data[std_sep_col]
    # # # Invert after thresholding: higher is better
    # # data['std/SEP'] = np.where(
    # #     data['std/SEP_raw'] >= threshold_raw,
    # #     1 - (data[std_sep_col] / data[std_sep_col].max()),
    # #     0  # Below threshold → 0
    # # )

    # Group by Crop and Freq
    # grouped = data.groupby(['Crop', 'Freq'])[['R²', 'SEP', 'std/SEP_raw', 'std/SEP', 'Bias']].mean().reset_index()
    grouped = data.groupby(['Crop', 'Freq'])[['R²', 'SEP', 'Bias']].mean().reset_index()
    categories = ['R²', 'SEP', 'Bias']

    # Plot
    for (crop, freq), group in grouped.groupby(['Crop', 'Freq']):
        values = group[categories].values.flatten()
        values = np.append(values, values[0])  # Close the circle

        angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
        angles += angles[:1]

        fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
        ax.fill(angles, values, color='red', alpha=0.25)
        ax.plot(angles, values, color='red', linewidth=2)

        # Add value labels
        values_flat = [group['R²'].iloc[0], group['SEP'].iloc[0], group['Bias'].iloc[0]]
        for angle, value, label in zip(angles[:-1], values_flat, categories):
            if label == "std/SEP":
                ax.text(angle, 0.9, f'{value:.3f}', ha='center', va='bottom', fontsize=10, color='red')
            else:
                ax.text(angle, value + 0.05, f'{value:.3f}', ha='center', va='bottom', fontsize=10, color='red')
        
        # threshold_norm = 1 - (threshold_raw / data[std_sep_col].max())
        # angles_thresh = [angles[2]] * 2  # Assuming 'std/SEP' is 3rd metric
        # values_thresh = [0, threshold_norm]
        # ax.plot(angles_thresh, values_thresh, color='gray', linestyle='--', linewidth=2)
        # ax.text(angles[2], threshold_norm + 0.05, f'Thresh: {threshold_norm:.2f}', color='gray', ha='center')

        ax.set_yticklabels([])
        ax.set_xticks(angles[:-1])
        ax.set_xticklabels(categories)
        plt.title(f"{comp} - {crop} ({freq})")
        plt.tight_layout()
        plt.show()   